In [9]:
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from skmultilearn.problem_transform import LabelPowerset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.metrics import roc_auc_score, plot_roc_curve, precision_score, recall_score
from sklearn.metrics import (confusion_matrix, precision_recall_curve, auc, roc_curve, recall_score, 
                             classification_report, f1_score, average_precision_score, precision_recall_fscore_support)
from sklearn import metrics
from sklearn.svm import SVC

In [2]:
df = pd.read_csv ('text_df.csv')
print(df)

           category                                               text  \
0          business  worldcom boss  left books alone  former worldc...   
1             sport  tigers wary of farrell  gamble  leicester say ...   
2             sport  yeading face newcastle in fa cup premiership s...   
3     entertainment  ocean s twelve raids box office ocean s twelve...   
4             sport  henman hopes ended in dubai third seed tim hen...   
...             ...                                                ...   
1402          sport  davies favours gloucester future wales hooker ...   
1403       business  beijingers fume over parking fees choking traf...   
1404       business  cars pull down us retail figures us retail sal...   
1405  entertainment  rem announce new glasgow concert us band rem h...   
1406          sport  souness delight at euro progress boss graeme s...   

      business  sport  entertainment  
0            1      0              0  
1            0      1            

In [3]:
print(df.business.value_counts())
print(df.sport.value_counts())
print(df.entertainment.value_counts())

0    897
1    510
Name: business, dtype: int64
0    896
1    511
Name: sport, dtype: int64
0    1021
1     386
Name: entertainment, dtype: int64


In [4]:
# Data Cleansing / Preprocessing
wordnet_lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()
stopwords = set(stopwords.words('english'))
def tokenize_lemma_stopwords(text):
    text = text.replace("\n", " ")
    # split string into words (tokens)
    tokens = nltk.tokenize.word_tokenize(text.lower())
    # keep strings with only alphabets
    tokens = [t for t in tokens if t.isalpha()]
    # put words into base form
    tokens = [wordnet_lemmatizer.lemmatize(t) for t in tokens] 
    tokens = [stemmer.stem(t) for t in tokens]
    cleanedText = " ".join(tokens)
    return cleanedText

def dataCleaning(df):
    data = df.copy()
    data["text"] =data["text"].apply(tokenize_lemma_stopwords)
    return data

# apply cleaning function to the data
cleaned_df = dataCleaning(df)
corpus = cleaned_df.text
corpus

0       worldcom bo left book alon former worldcom bo ...
1       tiger wari of farrel gambl leicest say they wi...
2       yead face newcastl in fa cup premiership side ...
3       ocean s twelv raid box offic ocean s twelv the...
4       henman hope end in dubai third seed tim henman...
                              ...                        
1402    davi favour gloucest futur wale hooker mefin d...
1403    beijing fume over park fee choke traffic jam i...
1404    car pull down u retail figur u retail sale fel...
1405    rem announc new glasgow concert u band rem hav...
1406    souness delight at euro progress bo graem soun...
Name: text, Length: 1407, dtype: object

In [12]:
#Feature Engineering
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(corpus)
y = df[['business','sport','entertainment']]

# Splitting data into train and test
X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                   test_size=0.25,
                                                   random_state=42)

svm = SVC(probability = True)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)
cr_svm = classification_report(y_test, y_pred_svm)
probs_svm = svm.predict_proba(X_test)
probs_svm = svm.predict_proba(X_test)
preds_svm =probs_svm[:,1]
print('Precision Score: ', round(precision_score(y_test, y_pred_svm), 2))
print('Recall Score: ', round(recall_score(y_test, y_pred_svm), 2))
print('F1 Score: ', round(f1_score(y_test, y_pred_svm), 2))
print('Accuracy Score: ', round(accuracy_score(y_test, y_pred_svm), 2))

Precision Score: 0.81
Recall Score: 0.93
F1 Score: 0.86
Accuracy Score: 0.85


In [6]:
def text_classifier():
    
    query = []
    user_text = input('Please enter text to be classified: ')
    query.append(user_text)
    
    #create dataframe of this text and clean it
    df_query = pd.DataFrame(query, columns=['text'])
    clean_query_df = dataCleaning(df_query)
    clean_query = clean_query_df.text.iloc[0]
    #Vectorizing the text
    vec_query = tfidf.transform([clean_query])
    
    # Make classifier and fit it to training data
    svm_classifier = LabelPowerset(LinearSVC(),require_dense = [False, True])
    svm_classifier.fit(X_train, y_train)
    
    # Predicting the class of user text using this model
    ans = svm_classifier.predict(vec_query).toarray()
    print(ans)
    if list(ans [0]) == [1,0,0]:
        result = 'business'
    elif list(ans [0]) == [0,1,0]:
        result = 'sport'
    elif list(ans [0]) == [0,0,1]:
        result = 'entertainment'
    else:
        print("This text does not belong to any of the categories.")
        
    return("The label is:", result)
        

In [7]:
text_classifier()

Please enter text to be classified: tigers wary of farrell  gamble  leicester say they will not be rushed
[[0 1 0]]


('The label is:', 'sport')

In [13]:
text_classifier()

Please enter text to be classified: ocean s twelve raids box office ocean s twelve  the crime
[[0 0 1]]


('The label is:', 'entertainment')

In [14]:
text_classifier()

Please enter text to be classified: worldcom boss  left books alone  former
[[0 1 0]]


('The label is:', 'sport')

In [15]:
text_classifier()

Please enter text to be classified: virgin blue shares plummet
[[1 0 0]]


('The label is:', 'business')